## Data Cleaning

In [37]:
# Moves CSV and Excel files into subfolders named from the text before the first space or underscore

# Limitation: North Queens land did not work due to space in the nmae.

from pathlib import Path
import re
import shutil

data_dir = Path(r"C:\Users\birad\Downloads\PS_Data_AI_Assignment2\feedback") # Please change this to the path of your data folder.
extensions = {".csv", ".xlsx", ".xls", ".xlsm", ".xlsb"}

for source in data_dir.iterdir():
    if not source.is_file() or source.suffix.lower() not in extensions:
        continue

    folder_name = re.split(r"[\s_]", source.stem, maxsplit=1)[0]
    if not folder_name:
        continue

    folder = data_dir / folder_name
    folder.mkdir(exist_ok=True)

    destination = folder / source.name
    if destination.exists():
        print(f"Skipped (already exists): {source.name}")
        continue

    shutil.move(str(source), str(destination))
    print(f"Moved: {source.name} → {folder_name}/")

In [38]:
# Skimmed files and manually removed blurbs before importing.

from pathlib import Path
import json
import pandas as pd

# Change data_dir to the folder containing your CSV and Excel files.
data_dir = Path(r"C:\Users\birad\Downloads\PS_Data_AI_Assignment2\feedback")
summary = {}

for file in sorted(data_dir.rglob("*")):
    if not file.is_file() or file.suffix.lower() not in {".csv", ".xlsx", ".xls"}:
        continue

    folder = str(file.parent.relative_to(data_dir))
    group = summary.setdefault(folder, {"files_ingested": 0, "files": []})

    try:
        if file.suffix.lower() == ".csv":
            try:
                df = pd.read_csv(file, encoding="utf-8-sig")
            except UnicodeDecodeError:
                df = pd.read_csv(file, encoding="cp1252")

            tables = {"CSV": df}
        else:
            tables = pd.read_excel(file, sheet_name=None)

        sheets = {}
        for name, df in tables.items():
            sheets[name] = {
                "rows": len(df),
                "column_names": [str(column) for column in df.columns],
                "unique_values": {
                    str(column): df[column].dropna().unique()[:10].tolist()
                    for column in df.columns
                },
            }

        group["files_ingested"] += 1
        group["files"].append({
            "file": file.name,
            "status": "success",
            "tables": sheets,
        })

    except Exception as error:
        group["files"].append({
            "file": file.name,
            "status": "failed",
            "error": str(error),
        })

output = json.dumps(summary, indent=2, default=str, ensure_ascii=False)
(data_dir / "import_summary.json").write_text(output, encoding="utf-8")
print(output)

{
  "Ballarat": {
    "files_ingested": 1,
    "files": [
      {
        "file": "Ballarat - Customer Satisfaction.xlsx",
        "status": "success",
        "tables": {
          "Responses": {
            "rows": 292,
            "column_names": [
              "Date",
              "Dept",
              "Name",
              "Phone",
              "How satisfied?",
              "Comments"
            ],
            "unique_values": {
              "Date": [
                "2025-10-01 00:00:00",
                "2025-10-02 00:00:00",
                "2025-10-09 00:00:00",
                "2025-10-10 00:00:00",
                "2025-10-11 00:00:00",
                "2025-10-16 00:00:00",
                "2025-10-18 00:00:00",
                "2025-10-21 00:00:00",
                "2025-10-22 00:00:00",
                "2025-10-23 00:00:00"
              ],
              "Dept": [
                "Parts",
                "Sales",
                "Service"
              ],
         

## Manually Dug through data

- Main themes: book system is not optimal, customer out reach is minimal, messages are not being passed to the relavent people. but there are instances where people report being kept in loop by sales rep or person in charge likely manually and through great effort
- Systems fail due to column name changes
- Taken by implies -> Bias
- some sheets have as issue of logic change ie 5 used to mean good now it means bad for rating
- some sheets have an issue of PII (Most) (customers leaving PII on feedback area and also being recoreded and sent by dealers)
- Frankston Ring Wood scroll down to see new data???
- One sheet just passed a piviot table.
- manually wrote down the column names corrosponding to departments cust_name, phone, customer feedback, keywords to hide PII in comments, date, Customer Rating, Rego, Taken By Email.

## Hide PII

In [39]:
# Golden rule here is to only work with PII that is encrypted like SHA 256 (did tihs in BBDO)
# But Apologies i do not have the time to do this. It is a long term solution



## Combine Data

In [40]:
from pathlib import Path
import re
import pandas as pd

# Change this path to your feedback folder.
# Ignore those 3 dealers for simplicity.
data_dir = Path(r"C:\Users\birad\Downloads\PS_Data_AI_Assignment2\feedback")
output_dir = data_dir.parent / "Feedback_sanitised"
output_dir.mkdir(exist_ok=True)

ignored_dealers = {"chatswood", "ringwood", "frankston"}
extensions = {".csv", ".xlsx", ".xls", ".xlsm"}

# Match common column names, ignoring case, punctuation and extra spaces.
column_patterns = {
    "Department": r"\b(?:dept|department|area)\b",
    "Comments": (
        r"\b(?:comments?|feedback|verbatim|notes?)\b"
        r"|\bcustomer\s+feed\s+back\b"
        r"|\bwhat\s+did\s+they\s+say\b"
        r"|\bcustomer\s+said\b"
    ),
    "Date": r"\bdate\b|\bsubmitted\s+at\b",
    "Rating": (
        r"\bhow\s+sat[ia]sfied\b"
        r"|\bcsat(?:\s+score)?\b"
        r"|\brating\b|\bhow\s+was\s+it\b|\bscore\b"
    ),
}

# These patterns mask common PII formats; review the output for anything missed.
pii_patterns = [
    # Email addresses.
    (r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", "[EMAIL]"),

    # Australian phone numbers, including +61 and spaced formats.
    (
        r"(?<!\w)(?:\+61[\s().-]*[23478]|0[23478])"
        r"(?:[\s().-]*\d){8}(?!\w)",
        "[PHONE]",
    ),
    (r"(?<!\w)(?:1300|1800)(?:[\s.-]*\d){6}(?!\w)", "[PHONE]"),
    (r"(?<!\w)13[\s.-]*\d{2}[\s.-]*\d{2}(?!\w)", "[PHONE]"),

    # Other phone-like numbers immediately following a contact request.
    (
        r"\b((?:call|phone|text|sms)\s+me(?:\s+(?:on|at))?\s*[:=-]?\s*)"
        r"\+?\d(?:[\d\s().-]{4,}\d)",
        r"\1[PHONE]",
    ),

    # Registrations explicitly labelled reg, rego, registration or number plate.
    (
        r"\b((?:reg(?:o|istration)?|number\s+plate)"
        r"(?:\s+(?:number|no\.?|is))?\s*[:=#(-]?\s*)"
        r"(?=[A-Z0-9 -]{2,10}(?!\w))"
        r"(?=[A-Z0-9 -]*\d)[A-Z0-9]{1,6}(?:[ -][A-Z0-9]{1,6})?",
        r"\1[REGISTRATION]",
    ),
]


def normalise_header(value):
    return re.sub(r"[^a-z0-9]+", " ", str(value).lower()).strip()


def sanitise_comment(value):
    if pd.isna(value):
        return pd.NA

    text = str(value)
    for pattern, replacement in pii_patterns:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text


def read_tables(file):
    # Inspect the first row as column headers.
    if file.suffix.lower() == ".csv":
        try:
            df = pd.read_csv(file, header=0, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(file, header=0, encoding="cp1252")
        return {"CSV": df}

    # Read every sheet in each Excel workbook.
    return pd.read_excel(file, sheet_name=None, header=0)


datasets = []
import_log = []

for file in sorted(data_dir.rglob("*")):
    if (
        not file.is_file()
        or file.suffix.lower() not in extensions
        or file.name.startswith("~$")
    ):
        continue

    relative_path = file.relative_to(data_dir)
    dealer = relative_path.parts[0] if len(relative_path.parts) > 1 else "unknown"

    if dealer.casefold() in ignored_dealers:
        continue

    try:
        tables = read_tables(file)
    except Exception as error:
        import_log.append({
            "Dealer": dealer,
            "SourceFile": str(relative_path),
            "Sheet": "",
            "Status": "failed",
            "Details": str(error),
        })
        continue

    for sheet_name, df in tables.items():
        df = df.dropna(how="all").reset_index(drop=True)
        if df.empty:
            continue

        cleaned = pd.DataFrame(index=df.index)
        missing_columns = []

        for target, pattern in column_patterns.items():
            matches = [
                column for column in df.columns
                if re.search(pattern, normalise_header(column), re.IGNORECASE)
            ]

            if matches:
                # If several columns match, take the first non-empty value per row.
                values = df[matches].replace(r"^\s*$", pd.NA, regex=True)
                cleaned[target] = values.bfill(axis=1).iloc[:, 0]
            else:
                cleaned[target] = pd.NA
                missing_columns.append(target)

        cleaned["Comments"] = cleaned["Comments"].map(sanitise_comment)

        # Keep original date and rating values for later standardisation.
        cleaned["Dealer"] = dealer
        cleaned["Sheet"] = str(sheet_name)
        cleaned["SourceFile"] = str(relative_path)

        # Remove rows with no values in any of the four requested fields.
        cleaned = cleaned.dropna(
            subset=["Department", "Comments", "Date", "Rating"],
            how="all",
        )
        datasets.append(cleaned)

        import_log.append({
            "Dealer": dealer,
            "SourceFile": str(relative_path),
            "Sheet": str(sheet_name),
            "Status": "imported",
            "Details": (
                f"Rows: {len(cleaned)}; "
                f"Missing columns: {', '.join(missing_columns) or 'none'}"
            ),
        })

final_columns = [
    "Department", "Comments", "Date", "Rating",
    "Dealer", "Sheet", "SourceFile",
]

final_dataset = (
    pd.concat(datasets, ignore_index=True)[final_columns]
    if datasets
    else pd.DataFrame(columns=final_columns)
)

final_dataset.to_csv(
    output_dir / "combined_feedback.csv",
    index=False,
    encoding="utf-8-sig",
)

# Log filenames and import results without copying customer comments.
pd.DataFrame(
    import_log,
    columns=["Dealer", "SourceFile", "Sheet", "Status", "Details"],
).to_csv(output_dir / "import_log.csv", index=False, encoding="utf-8-sig")

print(f"Saved {len(final_dataset):,} rows to {output_dir / 'combined_feedback.csv'}")

Saved 6,228 rows to C:\Users\birad\Downloads\PS_Data_AI_Assignment2\Feedback_sanitised\combined_feedback.csv


In [41]:
import pandas as pd
from pathlib import Path

csv_path = Path("Feedback_sanitised/combined_feedback.csv")
if not csv_path.exists():
    csv_path = Path("..") / csv_path

df = pd.read_csv(csv_path)
df.head()

,Department,Comments,Date,Rating,Dealer,Sheet,SourceFile
0,Parts,"Not cheap cheap, but fair, and for the actual ...",2025-10-01 00:00:00,Satisfied,Ballarat,Responses,Ballarat\Ballarat - Customer Satisfaction.xlsx
1,Sales,Moved here from interstate and was nervous byi...,2025-10-02 00:00:00,Satisfied,Ballarat,Responses,Ballarat\Ballarat - Customer Satisfaction.xlsx
2,Service,NaN,2025-10-02 00:00:00,Satisfied,Ballarat,Responses,Ballarat\Ballarat - Customer Satisfaction.xlsx
3,Service,WHY DO I ALWAYS HAVE TO BE THE ONE CHASING? EV...,2025-10-02 00:00:00,Dissatisfied,Ballarat,Responses,Ballarat\Ballarat - Customer Satisfaction.xlsx
4,Service,Day wasted.,2025-10-02 00:00:00,Very dissatisfied,Ballarat,Responses,Ballarat\Ballarat - Customer Satisfaction.xlsx


In [42]:
# similar departments should sart with teh same letters.

sorted(
    df["Department"].dropna().str.strip().unique(),
    key=str.casefold
)

['New Cars',
 'P',
 'P&A',
 'Parts',
 'Parts & Accessories',
 'parts counter',
 'S',
 'Sales',
 'sales',
 'Sales - New',
 'Sales - Used',
 'Service',
 'service',
 'SERVICE',
 'Service Dept',
 'SV',
 'Svc',
 'Used Cars',
 'Workshop']

In [43]:
department_mapping = {
    "new cars": "New Cars",
    "sales - new": "New Cars",
    "p": "Parts",
    "p&a": "Parts",
    "parts": "Parts",
    "parts & accessories": "Parts",
    "parts counter": "Parts",
    "sales": "Sales",
    "s": "Service",
    "service": "Service",
    "service dept": "Service",
    "sv": "Service",
    "svc": "Service",
    "used cars": "Used Cars",
    "sales - used": "Used Cars",
    "workshop": "Workshop",
}

df["Department"] = (
    df["Department"]
    .str.strip()
    .str.casefold()
    .replace(department_mapping)
)

In [ ]:

"""


## RAN this code ealier but i manually went through the excel file to make
## Sure Dates where properly formatted DO NOT RUN THIS ADITYA AGAIN.


from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, date
import re

original_dates = df["Date"].copy()

def clean_date(value):
    if pd.isna(value):
        return pd.NaT

    # Already a datetime — don't reinterpret it.
    if isinstance(value, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(value)

    text = str(value).strip()

    if not text:
        return pd.NaT

    # Excel serial numbers.
    if re.fullmatch(r"\d{5}(?:\.\d+)?", text):
        number = float(text)
        if 20000 <= number <= 80000:
            return pd.to_datetime(
                number, unit="D", origin="1899-12-30"
            )

    # STRICT Australian DD/MM/YYYY, including times.
    # Also handles single-digit days/months.
    for fmt in [
        "%d/%m/%Y %H:%M:%S",
        "%d/%m/%Y %H:%M",
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d/%m/%y",
        "%d-%m-%y",
        "%d-%b-%y",
        "%d-%b-%Y",
        "%d %b %Y",
        "%d %B %Y",
        "%Y-%m-%d",
        "%Y-%m-%d %H:%M:%S",
    ]:
        try:
            return pd.Timestamp(datetime.strptime(text, fmt))
        except ValueError:
            continue

    # Unrecognised or incomplete dates.
    return pd.NaT


# Create NEW column; original Date remains unchanged.
df["Date_cleaned"] = pd.to_datetime(
    original_dates.map(clean_date),
    errors="coerce"
).dt.strftime("%d/%m/%Y")

# Review failed conversions.
invalid_mask = (
    original_dates.notna()
    & original_dates.astype("string").str.strip().ne("")
    & df["Date_cleaned"].isna()
)

print("Unparsed dates:")
print(df.loc[invalid_mask, ["SourceFile", "Date"]].to_string())

print("\nSuccessfully converted:", df["Date_cleaned"].notna().sum())
print("Failed conversions:", invalid_mask.sum())

# Output folder.
base_dir = Path("Feedback_sanitised")
if not base_dir.exists():
    base_dir = Path("..") / base_dir

output_dir = base_dir / "one_date"
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(
    ## output_dir / "combined_feedback.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", output_dir / "combined_feedback.csv")


Unparsed dates:
                                                   SourceFile                                                                Date
2763                                IPS\IPS_feedback_log.xlsx                                                           early Oct
2780                                IPS\IPS_feedback_log.xlsx                                                             mid Nov
2827                                IPS\IPS_feedback_log.xlsx                                                           early Jan
2838                                IPS\IPS_feedback_log.xlsx                                                             wk1 Feb
2891                                IPS\IPS_feedback_log.xlsx                                                             wk5 Mar
2908                                IPS\IPS_feedback_log.xlsx                                                           early Apr
2946                                IPS\IPS_feedback_log.xlsx             

## NLP Theme classification (Cleaning)

In [45]:
# Using the cleaned date column, formated departments, dealers (file names from ealier)
## did not clean ratings. People can have mixed feelings Neutral but identify critical issues in system

In [1]:
import pandas as pd

path = r"C:\Users\birad\Downloads\PS_Data_AI_Assignment2\Feedback_sanitised\one_date\combined_feedback.csv"
df = pd.read_csv(path)

# Check Date_cleaned (day/month/year)
dates = pd.to_datetime(df["Date_cleaned"], format="%d/%m/%Y", errors="coerce")

print("Missing dates:", df["Date_cleaned"].isna().sum())
print("Invalid dates:")
print(df.loc[df["Date_cleaned"].notna() & dates.isna(), ["Date", "Date_cleaned"]])
print("Date range:", dates.min(), "to", dates.max())

# Print unique values only
print("\nDepartment unique:")
print(df["Department"].unique())

print("\nDealer unique:")
print(df["Dealer"].unique())

Missing dates: 2
Invalid dates:
Empty DataFrame
Columns: [Date, Date_cleaned]
Index: []
Date range: 2025-10-01 00:00:00 to 2026-11-26 00:00:00

Department unique:
<StringArray>
['Service', 'Sales', 'Parts', 'Workshop', nan, 'Used Cars', 'New Cars']
Length: 7, dtype: str

Dealer unique:
<StringArray>
[ 'liverpool', 'Shepparton',    'bendigo',    'boxhill',  'Chermside',
   'ESSENDON',    'Geelong',  'North QLD', 'Parramatta',    'Penrith',
  'Southport', 'Springwood',   'Tamworth',  'Toowoomba', 'Wollongong',
  'Dandenong',     'Hunter',        'IPS',      'Noosa',   'Ballarat']
Length: 20, dtype: str


In [2]:
# Keep dates from Q3 2025 through the end of Q3 2026
dates = pd.to_datetime(df["Date_cleaned"], format="%d/%m/%Y", errors="coerce")

keep = dates.between("2025-07-01", "2026-09-30")
print("Rows removed (outside range or missing/invalid dates):", (~keep).sum())

df = df.loc[keep].copy()

print("Date range:", dates[keep].min(), "to", dates[keep].max())

Rows removed (outside range or missing/invalid dates): 6
Date range: 2025-10-01 00:00:00 to 2026-09-30 00:00:00


In [ ]:
# Due to the PII issue leaking into customer feedback I will not use LLMs to do NLP (too high risk does not help with making final decision)
# i will need to use a simpler key word matching approach.



In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 6199 entries, 0 to 6202
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Department    5853 non-null   str  
 1   Comments      4685 non-null   str  
 2   Date          6199 non-null   str  
 3   Rating        6189 non-null   str  
 4   Dealer        6199 non-null   str  
 5   Sheet         6199 non-null   str  
 6   SourceFile    6199 non-null   str  
 7   Date_cleaned  6199 non-null   str  
dtypes: str(8)
memory usage: 435.9 KB


In [ ]:
import re
import pandas as pd

comment_col = "Comments"

# Broad search: possible matches for manual review.
keywords = [
    r"\bcall(?:ed|ing)?\b",
    r"\bcall\s*back\b",
    r"\bring(?:ing)?\b",
    r"\brang\b",
    r"\brung\b",
    r"\bphone(?:d)?\b",
    r"\bcontact(?:ed|ing)?\b",
    r"\bget(?:ting)? back\b",
    r"\bgot back\b",
    r"\bhear(?:d)? back\b",
    r"\bfollow[\s-]?up\b",
    r"\brepl(?:y|ied|ies)\b",
    r"\brespon(?:se|ses|d|ded|ding)\b",
    r"\breturn(?:ed|ing)?\b.{0,20}\bcall",
    r"\bwait(?:ed|ing)?\b",
    r"\bwast(?:e|ed|ing)\b",
    r"\bsms\b",
    r"\btext(?:ed|ing|s)?\b",
    r"\bemail(?:ed|ing|s)?\b",
    r"\bmessage(?:s|d)?\b",
    r"\bsystem\b",
    r"\bwebsite\b",
    r"\bapp\b",
    r"\binfo(?:rmation)?\b",
    r"\bdelay(?:ed|s)?\b",
    r"\bupdat(?:e|es|ed|ing)\b",
    r"\bchase(?:d|s)?\b",
    r"\bchasing\b",
    r"\bno[\s-]?one\b",
    r"\bnobody\b",
]

pattern = re.compile("|".join(keywords), flags=re.IGNORECASE)
comments = df[comment_col].fillna("").astype(str)

mask = comments.str.contains(pattern, na=False)
examples = df.loc[mask].copy()

examples["matched_keywords"] = [
    ", ".join(sorted({match.group(0).lower() for match in pattern.finditer(text)}))
    for text in comments.loc[mask]
]

print(f"{len(examples):,} possible matches out of {len(df):,} rows")

# Display all matches with full text and all original columns.
with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(examples)

# Save all matching examples.
examples.to_csv("callback_keyword_examples.csv", index=False)

In [8]:
# Keep only rows that did NOT match the keyword search.
excluded_results = df.loc[~mask].copy()

print(f"{len(excluded_results):,} rows remaining after excluding keyword matches")

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(excluded_results)

excluded_results.to_csv("comments_without_keyword_matches.csv", index=False)

4,547 rows remaining after excluding keyword matches


,Department,Comments,Date,Rating,Dealer,Sheet,SourceFile,Date_cleaned
0,Service,"No jargon, no upsell, just a proper explanation. Then the check engine light was back by Monday.",8/12/2026,3,liverpool,CSV,liverpool\liverpool_q3_2026.csv,12/08/2026
1,Service,NaN,1/12/2026,5,Shepparton,Log,Shepparton\Shepparton feedback FINAL - site closed.xlsx,12/01/2026
4,Service,I'm not one for fuss and neither was this. Easy from the get-go.,9/11/2026,4,liverpool,CSV,liverpool\liverpool_q3_2026.csv,11/09/2026
5,Sales,NaN,7/11/2026,5,liverpool,CSV,liverpool\liverpool_q3_2026.csv,11/07/2026
8,Service,I have recently moved to this area and was welcomed very warmly by the service staff. They were courteous and patient. I will be returning. (rego [REGISTRATION]),9/10/2026,5,liverpool,CSV,liverpool\liverpool_q3_2026.csv,10/09/2026
10,Sales,"Great deal for the money, would pay it again.",9/10/2026,4,liverpool,CSV,liverpool\liverpool_q3_2026.csv,10/09/2026
11,Workshop,"They wanted $90 for a diagnostic when the fault code was obvious, and then charged me OEM price for a part that sat on back-order for weeks. Quote said aftermarket. Not good enough.",6/10/2026,1,Shepparton,Log,Shepparton\Shepparton feedback FINAL - site closed.xlsx,10/06/2026
12,Service,"Lovely greeting, lovely manner Aroha was a real pleasure",30/09/2026,4,bendigo,CSV,bendigo\bendigo feedback.csv,30/09/2026
13,Service,Still pinging on acceleration. Third visit.,30/09/2026,1,bendigo,CSV,bendigo\bendigo feedback.csv,30/09/2026
14,Service,Was told I'd get a full demo of charging. Didn't happen.,30/09/2026,1,boxhill,CSV,boxhill\boxhill_survey_export_2026Q3.csv,30/09/2026
